# VSI-590K + SPAR posed manifests: review

Read-only check of the manifests from `jepa3d/trainbatch/make_mix_manifests.sbatch`
(design: `jepa3d/docs/train_data_vsi_spar.md`).

1. Overview: rows, scenes, types, answer repetition
2. Samples: query / target / answer_key + the N views + camera layout
3. Frame quality: what the filter drops
4. InfoNCE: same-answer pairs per batch, by exact text vs `answer_key`

In [ ]:
import json, random, sys
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

ROOT = Path.cwd().parent.parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
from jepa3d.frame_pool import keep_mask, load_pool
from jepa3d.paths import VJEPA_ROOT

N = 8
DATA = ROOT / "data"
FILES = {
    "vsi590k":    DATA / f"vsi590k_sn_f{N}.jsonl",
    "spar_train": DATA / f"spar_video_train_f{N}.jsonl",
    "spar_val":   DATA / f"spar_video_val_f{N}.jsonl",
}
MAX_ROWS = 200_000   # reservoir sample per file
SEED = 0

def load(path, k=MAX_ROWS, seed=SEED):
    rng, keep, seen = random.Random(seed), [], 0
    with open(path) as f:
        for line in f:
            seen += 1
            if len(keep) < k:
                keep.append(line)
            else:
                j = rng.randrange(seen)
                if j < k:
                    keep[j] = line
    return [json.loads(l) for l in keep], seen

data = {}
for name, path in FILES.items():
    data[name], total = load(path)
    print(f"{name:11s} {total:>9,} rows  (read {len(data[name]):,})")
stats = json.load(open(DATA / f"mix_vsi_spar_train_f{N}.jsonl.stats.json"))
print("mix:", stats["rows"], "rows,", stats["scenes"], "scenes", dict(stats["by_source"]))

## 1. Overview

`top_key%` = share of the most common `answer_key` in the type; `uniq_key%` = distinct keys / rows.

In [ ]:
for name, rows in data.items():
    print(f"\n== {name}")
    print(f"{'type':38s} {'rows':>8s} {'scenes':>7s} {'uniq_key%':>9s} {'top_key%':>8s}  top key")
    by = defaultdict(list)
    for r in rows:
        by[r["question_type"]].append(r)
    for t, rs in sorted(by.items(), key=lambda kv: -len(kv[1])):
        keys = Counter(r["answer_key"] for r in rs)
        top, n = keys.most_common(1)[0]
        print(f"{t:38s} {len(rs):>8,} {len({(r['subset'], r['scene']) for r in rs}):>7,} "
              f"{100*len(keys)/len(rs):8.1f}% {100*n/len(rs):7.1f}%  {top[:50]!r}")
    print("views/row:", dict(Counter(len(r["images"]) for r in rows)),
          "| subsets:", dict(Counter(r["subset"] for r in rows)))

## 2. Samples

Per type: text fields, the N views (time order), and a top-down camera plot (grey = whole scene pool, red = chosen views, arrow = viewing direction).

In [ ]:
SPAR_ROOT = VJEPA_ROOT / "source_data/spar"
POSED_ROOT = VJEPA_ROOT / "source_data/scannet/posed_images"
_pools = {}

def pool_of(r):
    key = (r["subset"], r["scene"])
    if key not in _pools:
        _pools[key] = load_pool(*key, SPAR_ROOT, POSED_ROOT)
    return _pools[key]

def show(r, thumb=200):
    print(f"[{r['source']} / {r['question_type']}]  {r['subset']}/{r['scene']}  frames {r['frame_ids']}")
    print("  query     :", r["query"])
    print("  target    :", r["target"])
    if r.get("target_short"):
        print(f"  short     : {r['target_short']}  ({r['target_style']})")
    print("  answer_raw:", r["answer_raw"][:300])
    print("  answer_key:", r["answer_key"])
    if r.get("options"):
        print("  options   :", r["options"])
    n = len(r["images"])
    fig = plt.figure(figsize=(2.2 * n + 3, 2.4))
    for i, p in enumerate(r["images"]):
        ax = fig.add_subplot(1, n + 1, i + 1)
        ax.imshow(Image.open(p)); ax.set_title(str(r["frame_ids"][i]), fontsize=8); ax.axis("off")
    ax = fig.add_subplot(1, n + 1, n + 1)
    pool = np.stack([f.c2w for f in pool_of(r).frames])
    ax.scatter(pool[:, 0, 3], pool[:, 1, 3], s=3, c="0.7")
    sel = np.array(r["c2w"])
    ax.quiver(sel[:, 0, 3], sel[:, 1, 3], sel[:, 0, 2], sel[:, 1, 2], color="r", scale=12)
    ax.set_aspect("equal"); ax.set_xticks([]); ax.set_yticks([]); ax.set_title("cams (x-y)", fontsize=8)
    plt.tight_layout(); plt.show()

PER_TYPE = 2
rng = random.Random(SEED)
for name in ("spar_train", "spar_val", "vsi590k"):
    by = defaultdict(list)
    for r in data[name]:
        by[r["question_type"]].append(r)
    print(f"\n{'=' * 30} {name} {'=' * 30}")
    for t, rs in sorted(by.items()):
        for r in rng.sample(rs, min(PER_TYPE, len(rs))):
            show(r)

## 3. Frame quality filter

Distribution of per-frame stats and the frames the default thresholds drop (`valid_frac >= 0.5`, `median_depth >= 0.6 m`, sharpness >= 0.3 x scene median).

In [ ]:
fs = {s: [json.loads(l) for l in open(DATA / f"frame_stats/{s}.jsonl")] for s in ("scannet", "scannetpp")}
fig, axes = plt.subplots(2, 3, figsize=(14, 6))
for row, (subset, st) in zip(axes, fs.items()):
    for ax, key in zip(row, ("valid_frac", "median_depth", "sharpness")):
        ax.hist([s[key] for s in st], bins=60); ax.set_title(f"{subset}: {key}"); ax.set_yscale("log")
plt.tight_layout(); plt.show()

for subset, st in fs.items():
    by_scene = defaultdict(list)
    for s in st:
        by_scene[s["scene"]].append(s)
    drop = [s for v in by_scene.values() for s, k in zip(v, keep_mask(v)) if not k]
    print(f"{subset}: dropped {len(drop):,} / {len(st):,} frames ({100*len(drop)/len(st):.1f}%)")
    pick = random.Random(SEED).sample(drop, min(8, len(drop)))
    fig, axes = plt.subplots(1, len(pick), figsize=(2.4 * len(pick), 2.4), squeeze=False)
    for ax, s in zip(axes[0], pick):
        f = next(f for f in load_pool(subset, s["scene"], SPAR_ROOT, POSED_ROOT).frames if f.frame_id == s["frame_id"])
        ax.imshow(Image.open(f.image)); ax.axis("off")
        ax.set_title(f"v{s['valid_frac']:.2f} d{s['median_depth']:.1f}", fontsize=8)
    plt.tight_layout(); plt.show()

## 4. InfoNCE: same-answer pairs in a batch

Random global batches from the mix (B = 32 per GPU x 3 GPUs = 96). For each anchor, count other rows with
the same exact target text (already identical Y-embeddings) and with the same `answer_key`
(also catches paraphrases). Both get masked in training; same-scene pairs with different answers stay negatives.

In [ ]:
mix, _ = load(DATA / f"mix_vsi_spar_train_f{N}.jsonl")
norm = lambda s: " ".join(s.lower().split()).rstrip(".")
text = np.array([hash(norm(r["target"])) for r in mix])
akey = np.array([hash(r["answer_key"]) for r in mix])
scene = np.array([hash(r["group_id"]) for r in mix])
qtype = np.array([r["source"] + "/" + r["question_type"] for r in mix])

def same_count(codes):
    _, inv, cnt = np.unique(codes, return_inverse=True, return_counts=True)
    return cnt[inv] - 1

B, TRIALS = 96, 2000
rs = np.random.default_rng(SEED)
hit = defaultdict(lambda: np.zeros(4))
for _ in range(TRIALS):
    idx = rs.choice(len(mix), B, replace=False)
    t, k, s = same_count(text[idx]), same_count(akey[idx]), same_count(scene[idx])
    for q, a, b, c in zip(qtype[idx], t > 0, k > 0, s > 0):
        hit[q] += (a, b, c, 1)
print(f"{'type':48s} {'anchors w/ same text':>20s} {'same answer_key':>16s} {'same scene':>11s}")
for q in sorted(hit, key=lambda q: -hit[q][1] / hit[q][3]):
    total = hit[q][3]
    print(f"{q:48s} {100*hit[q][0]/total:19.1f}% {100*hit[q][1]/total:15.1f}% {100*hit[q][2]/total:10.1f}%")